In [ ]:
using ProgressBars, Bessels
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 4*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift_b = - 0.05 #Lamb shift in GHz for the mode b
Lamb_shift_a = - 0.0 #Lamb shift in GHz for the mode a

ω_b = ω_b_0 #- Lamb_shift*2*pi # Shifted frequency of mode b

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ_b=1/10; # 1/ns from Leon's paper
ϕ_quantum=1/2

ω_d=ω_b_0 + Lamb_shift_b*2*pi# Drive frequency
ω_p=2ω_a-ω_d # Pump Frequency

ϵ_0 = 0.6*pi
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=0.75*g2_max

κ_2 = 4*g2_max^2/κ
κ_a = 3e-3*κ_2; #from Leon's paper

In [ ]:
function Find_Lamb_shift(params_chain,params_circuit; modes_below=false, normal_modes_distribution=false)
    
    onsite_energies,hopping_energies,system_chain_coupling = params_chain
    n=length(onsite_energies[1]) - 1 # number of oscillators in the chain
    C_sys_inv, ω_sys = params_circuit
    Z_sys = C_sys_inv/ω_sys #ħ/e^2


    ### Mapping ###
    ω = vcat(ω_sys,onsite_energies) # GHz
    g_q=vcat(0, -hopping_energies) 
    g_ϕ=vcat(-system_chain_coupling*sqrt(Z_sys), hopping_energies) 
    
    Mq=SymTridiagonal(ω, g_q)
    Mϕ=SymTridiagonal(ω, g_ϕ)

    A=Mq * Mϕ
    vals, vecs = eigen(A)

    ωk² = real(vals)
    ωk = sqrt.(ωk²)
    
    P = real(vecs)  # columns = normal modes
    ω_related_ls = ωk[argmax(abs.(P[1, :]))]
    fidelity = maximum(abs.(P[1, :]))^2/sum(P[1, :].^2) # Fidelity of the first mode 
    output=[]
    push!(output, ω_sys-ω_related_ls, fidelity)
    #output=[(ω_sys -  ω_related_ls),(ω_sys -  test), fidelity]
    if modes_below
        nb_modes_below_sys= sum(ωk .< ω_sys)
        push!(output, nb_modes_below_sys)
    end
    if normal_modes_distribution
        # Return the distribution of normal modes
        push!(output, ωk)
    end
    return output
end;

## Ohmic bath

In [ ]:
N=60
n_ohm=2000
Lamb_shift_ohm=zeros(N)
Fidelity_ohm = zeros(N)
modes_below_ohm = zeros(N)
ω_k_list_ohm = Vector{Vector{Float64}}(undef, N)
ω_cut_off_list_ohm=zeros(N)
ω_cut_off_min=19.95*2*pi
ω_cut_off_max=20.05*2*pi
for i in ProgressBar(1:N)
    ω_cut_off = ω_cut_off_min + (ω_cut_off_max - ω_cut_off_min) * (i-1) / (N-1)
    ω_cut_off_list_ohm[i] = ω_cut_off
    AB,J=Ohmic_bath(;bath_params=(ωc_dwn=0.0*2*pi, ωc_up=ω_cut_off), κ=κ_b, C_inv=C_b_inv)
    chain_params = chaincoeffs_finiteT(n_ohm, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    #println(Find_Lamb_shift(params_chain, (C_sys_inv, ω_sys), modes_below=true, normal_modes_distribution=true))
    Lamb_shift_ohm[i],Fidelity_ohm[i],modes_below_ohm[i],ω_k_list_ohm[i]= Find_Lamb_shift(chain_params, (C_b_inv, ω_b_0), modes_below=true, normal_modes_distribution=true)
end



In [ ]:
#ls_theo_ohm=(2κ/C_sys_inv.*(ω_cut_off_list_ohm))*Z_sys/2 # κ*ωc/ω_sys
#ls_test =@. ω_sys * sqrt(1+4*Z_sys*2κ/C_sys_inv*(ωc_list)/pi)-ω_sys
AB,J=Ohmic_bath(;bath_params=(ωc_dwn=0.0*2*pi, ωc_up=20*2*pi), κ=κ_b, C_inv=C_b_inv)
ls_theo_ohm=Compute_lamb_shift(J,20*2*pi,ω_b_0, Z_b)
#ls_theo_ohm= @.(κ_b*Z_b/(C_b_inv)*(ω_cut_off_list_ohm+ω_b_0*log(ω_cut_off_list_ohm/ω_b_0-1)))/2*pi;

In [ ]:
-ls_theo_test./ω_b_0*1e3*pi*pi

In [ ]:
simulation_cut_off_freq=20*2*pi

p1=plot((ω_cut_off_list_ohm.-simulation_cut_off_freq)./ω_b_0 *1e3,-(ω_b_0.-getindex.(ω_k_list_ohm, 300))./ω_b_0 *1e3, label="", linecolor=Perso_palette_hex[2])
for i in 900:1000#length(ω_k_list_ohm[1])
    if i%2==0
        col= Perso_palette_hex[2]
    else
        col= Perso_palette_hex[1]
    end
    plot!(((ω_cut_off_list_ohm.-simulation_cut_off_freq)./ω_b_0)*1e3,-(ω_b_0.-getindex.(ω_k_list_ohm, i))./ω_b_0 *1e3, label="", linecolor=col)
end
scatter!(((ω_cut_off_list_ohm.-simulation_cut_off_freq)./ω_b_0)*1e3, -Lamb_shift_ohm./ω_b_0 *1e3, label="",color=:black, alpha=Fidelity_ohm)
plot!(((ω_cut_off_list_ohm.-simulation_cut_off_freq)./ω_b_0)*1e3,-ls_theo_ohm./ω_b_0*1e3, label="Theoretical shift", linecolor=Perso_palette_hex[4])
vline!([0], label="", linecolor=:red, linestyle=:dash)
plot!(xlabel = string(L"(\omega_c-\omega_{c,simu})/\omega_{b,0}~(10^{-3})"), ylabel = string("Lamb shift ",L"\Delta\omega_{LS}/\omega_{b,0}~(10^{-3})"), legend=:bottomleft, ylims=(-13,-7))
plot(p1)
#savefig("../Figures/Lamb_shift/Lamb_shift_simu_two_photon_ohm.pdf")

In [ ]:
ω_interest=  20*2*pi
Lamb_shift_ohm_n=zeros(length(100:10:1500))

AB,J = Ohmic_bath(;bath_params=(ωc_dwn=0.0*2*pi, ωc_up=ω_interest), κ=κ_b, C_inv=C_b_inv)
for n_i in ProgressBar(100:10:1500)
    i= Int((n_i-100)/10 + 1)
    params_chain=chaincoeffs_finiteT(n_i, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    Lamb_shift_ohm_n[i],_= Find_Lamb_shift(params_chain, (C_b_inv, ω_b_0))
end


In [ ]:
ls_theo_ohm

In [ ]:
ω_0 = 50*2*pi
p1=plot(100:10:1500, Lamb_shift_ohm_n./ω_b_0, linewidth=.5, label="Numerical shift",
     marker=(:circle,1))
hline!([ls_theo_ohm/ω_b_0], label="Theoretical shift", color=:red, linestyle=:dash, linewidth=1)
plot!(xlabel = "Number of oscillators in the chain", ylabel = string("Lamb shift ",L"\Delta\omega/\omega_{sys}"), legend=:bottomleft)

p2=plot(100:10:1500, abs.(Lamb_shift_ohm_n.-ls_theo_ohm)/(ls_theo_ohm), linewidth=.5, label="",
     marker=(:circle,1))
plot!(xlabel = "Number of oscillators in the chain", ylabel = "Relative error on Lamb shift", yaxis=:log10)

plot(p1,p2, layout=(1,2), size=(2*size_px[1],size_px[2]))
#savefig("Lamb_shift_n.pdf")

In [ ]:
55/9.1

In [ ]:
85/12.4

## Ohmic bath with exponential decay

In [ ]:
N=80
n_exp=2000
Lamb_shift_e=zeros(N)
Fidelity_e = zeros(N)
modes_below_e = zeros(N)
ω_k_list_e = Vector{Vector{Float64}}(undef, N)
ω_cut_off_list_e=zeros(N)
ω_cut_off_min=19.95*2*pi
ω_cut_off_max=20.05*2*pi
for i in ProgressBar(1:N)
    ω_cut_off = ω_cut_off_min + (ω_cut_off_max - ω_cut_off_min) * (i-1) / (N-1)
    ω_cut_off_list_e[i] = ω_cut_off
    AB,J=Exp_decay(;bath_params=(ω_0=50*2*pi, ω_cut_off=ω_cut_off), κ=κ_b, C_inv=C_b_inv)
    chain_params = chaincoeffs_finiteT(n_exp, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    #println(Find_Lamb_shift(params_chain, (C_sys_inv, ω_sys), modes_below=true, normal_modes_distribution=true))
    Lamb_shift_e[i],Fidelity_e[i],modes_below_e[i],ω_k_list_e[i]= Find_Lamb_shift(chain_params, (C_b_inv, ω_b_0), modes_below=true, normal_modes_distribution=true)
end


In [ ]:
ω_0 = 50*2*pi # Characteristic frequency of the exponential decay in GHz

ls_theo_e=@. (2κ/C_sys_inv.*ω_c*(1-exp(-ω_cut_off_list_e/ω_c)))*Z_sys/2;

In [ ]:
using QuadGK
function cauchy_quadgk(g, a, b; kws...)
    a < 0 < b || throw(ArgumentError("domain must include 0"))
    g₀ = g(0)
    g₀int = b == -a ? zero(g₀) : g₀ * log(abs(b/a)) / (b - a)
    return quadgk_count(x -> (g(x)-g₀)/x + g₀int, a, 0, b; kws...)
end
function Compute_lamb_shift(J,i,ω_cut_off,ω_sys, Z_sys)
    J_shift(ω)=J(ω+ω_sys,i)
    return cauchy_quadgk(J_shift, -ω_sys, ω_cut_off-ω_sys)[1]* Z_sys/(4π)*2*pi
end


AB,J=Exp_decay(;bath_params=(ω_0=50*2*pi, ω_cut_off=20*2*pi), κ=κ_b, C_inv=C_b_inv)
ls_theo_e=Compute_lamb_shift(J,2,20*2*pi,ω_b_0, Z_b)


In [ ]:
simulation_cut_off_freq=20*2*pi

p1=plot((ω_cut_off_list_e.-simulation_cut_off_freq)./ω_b_0 *1e3,-(ω_b_0.-getindex.(ω_k_list_e, 300))./ω_b_0 *1e3, label="", linecolor=Perso_palette_hex[2])
for i in 900:1000#length(ω_k_list_e[1])
    if i%2==0
        col= Perso_palette_hex[2]
    else
        col= Perso_palette_hex[1]
    end
    plot!(((ω_cut_off_list_e.-simulation_cut_off_freq)./ω_b_0)*1e3,-(ω_b_0.-getindex.(ω_k_list_e, i))./ω_b_0 *1e3, label="", linecolor=col)
end
scatter!(((ω_cut_off_list_e.-simulation_cut_off_freq)./ω_b_0)*1e3, -Lamb_shift_e./ω_b_0 *1e3, label="",color=:black, alpha=Fidelity_e)
#plot!(ω_cut_off_list_ohm./ω_b_0,ls_theo_ohm./ω_b_0, label="Theoretical shift", linecolor=Perso_palette_hex[4])
vline!([0], label="", linecolor=:red, linestyle=:dash)
plot!(xlabel = string(L"(\omega_c-\omega_{c,simu})/\omega_{b,0}~(10^{-3})"), ylabel = string("Lamb shift ",L"\Delta\omega_{LS}/\omega_{b,0}~(10^{-3})"), legend=:bottomleft, ylims=(-9,-6))
plot(p1)
#savefig("../Figures/Lamb_shift/Lamb_shift_simu_two_photon.pdf")

In [ ]:
p1=scatter(ω_cut_off_list_e./ω_sys, Lamb_shift_e./ω_sys, markersize=2, label="")
plot!(ω_cut_off_list_e./ω_sys,ls_theo_e./ω_sys, linewidth=1, label="Theoretical shift")
plot!(xlabel = string(L"ω_c/ω_{sys}"), ylabel = string("Lamb shift ",L"\Delta\omega/\omega_{sys}"), legend=:bottomleft)
vline!([ω_cut_off_list_e[findall(modes_below .== 192)]/ω_sys], label="", color=:red, linestyle=:dash)
p2=scatter(ω_cut_off_list_e./ω_sys, modes_below, markersize=2, label="")
plot!(xlabel = string(L"ω_c/ω_{sys}"), ylabel = "Number of modes below "*L"ω_{sys}")
vline!([ω_cut_off_list_e[findall(modes_below .== 192)]/ω_sys], label="", color=:red, linestyle=:dash)
plot(p1,p2, layout=(1,2),size=(600,300))
#savefig("Lamb_shift_avoided_crossing.pdf")

In [ ]:
findall(Modes_below_sys .== 192)

In [ ]:
ω_interest=  19.998*2*pi
Lamb_shift_e_n=zeros(length(100:100:2000))

AB,J=Exp_decay(;bath_params=(ω_0=50*2*pi, ω_cut_off=ω_interest), κ=κ_b, C_inv=C_b_inv)
for n_i in ProgressBar(100:100:2000)
    i= Int((n_i-100)/100 + 1)
    params_chain=chaincoeffs_finiteT(n_i, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    Lamb_shift_e_n[i],_= Find_Lamb_shift(params_chain, (C_b_inv, ω_b_0))
end


In [ ]:
ls_theo_e/ω_b_0

In [ ]:
ω_0 = 50*2*pi
p1=plot(100:100:2000, Lamb_shift_e_n./ω_b_0, linewidth=.5, label="Numerical shift",
     marker=(:circle,1))
hline!([ls_theo_e/ω_b_0], label="Theoretical shift", color=:red, linestyle=:dash, linewidth=1)
plot!(xlabel = "Number of oscillators in the chain", ylabel = string("Lamb shift ",L"\Delta\omega/\omega_{sys}"), legend=:bottomleft)

p2=plot(100:100:2000, abs.(Lamb_shift_e_n.-ls_theo_e)/(ls_theo_e), linewidth=.5, label="",
     marker=(:circle,1))
plot!(xlabel = "Number of oscillators in the chain", ylabel = "Relative error on Lamb shift", yaxis=:log10)

plot(p1,p2, layout=(1,2), size=(2*size_px[1],size_px[2]), left_margin=10Plots.mm)
#savefig("Lamb_shift_n.pdf")

## Drude-Lorentz

In [ ]:
N=80
n_l=2000
Lamb_shift_l=zeros(N)
Fidelity_l = zeros(N)
modes_below_l = zeros(N)
ω_k_list_l = Vector{Vector{Float64}}(undef, N)
ω_cut_off_list_l=zeros(N)
ω_cut_off_min=19.95*2*pi
ω_cut_off_max=20.05*2*pi
for i in ProgressBar(1:N) #wrap any iterator
    ω_cut_off = ω_cut_off_min + (ω_cut_off_max - ω_cut_off_min) * (i-1) / (N-1)
    ω_cut_off_list_l[i] = ω_cut_off
    AB,J=Drude_lorentz(;bath_params=(Ω=50*2*pi, ω_cut_off=ω_cut_off), κ=κ_b, C_inv=C_b_inv)
    chain_params = chaincoeffs_finiteT(n_l, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    Lamb_shift_l[i],Fidelity_l[i],modes_below_l[i],ω_k_list_l[i]= Find_Lamb_shift(chain_params, (C_b_inv, ω_b_0), modes_below=true, normal_modes_distribution=true)
end

In [ ]:
AB,J=Drude_lorentz(;bath_params=(Ω=50*2*pi, ω_cut_off=20*2*pi), κ=κ_b, C_inv=C_b_inv)
ls_theo_l=Compute_lamb_shift(J,20*2*pi,ω_b_0, Z_b)

In [ ]:
simulation_cut_off_freq=20*2*pi

p1=plot((ω_cut_off_list_l.-simulation_cut_off_freq)./ω_b_0 *1e3,-(ω_b_0.-getindex.(ω_k_list_l, 300))./ω_b_0 *1e3, label="", linecolor=Perso_palette_hex[2])
for i in 900:1000#length(ω_k_list_l[1])
    if i%2==0
        col= Perso_palette_hex[2]
    else
        col= Perso_palette_hex[1]
    end
    plot!(((ω_cut_off_list_l.-simulation_cut_off_freq)./ω_b_0)*1e3,-(ω_b_0.-getindex.(ω_k_list_l, i))./ω_b_0 *1e3, label="", linecolor=col)
end
scatter!(((ω_cut_off_list_l.-simulation_cut_off_freq)./ω_b_0)*1e3, -Lamb_shift_l./ω_b_0 *1e3, label="",color=:black, alpha=Fidelity_l)
#plot!(ω_cut_off_list_ohm./ω_b_0,ls_theo_ohm./ω_b_0, label="Theoretical shift", linecolor=Perso_palette_hex[4])
vline!([0], label="", linecolor=:red, linestyle=:dash)
plot!(xlabel = string(L"(\omega_c-\omega_{c,simu})/\omega_{b,0}~(10^{-3})"), ylabel = string("Lamb shift ",L"\Delta\omega_{LS}/\omega_{b,0}~(10^{-3})"), legend=:bottomleft, ylims=(-10,-7))
plot(p1)
#savefig("../Figures/Lamb_shift/Lamb_shift_simu_two_photon.pdf")

In [ ]:
ω_interest=  20*2*pi
Lamb_shift_l_n=zeros(length(100:50:2000))

AB,J=Drude_lorentz(;bath_params=(Ω=50*2*pi, ω_cut_off=ω_interest), κ=κ_b, C_inv=C_b_inv)
for n_i in ProgressBar(100:50:2000)
    i= Int((n_i-100)/50 + 1)
    params_chain=chaincoeffs_finiteT(n_i, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    Lamb_shift_l_n[i],_= Find_Lamb_shift(params_chain, (C_b_inv, ω_b_0))
end


In [ ]:
ω_0 = 50*2*pi
p1=plot(100:50:2000, Lamb_shift_l_n./ω_b_0, linewidth=.5, label="Numerical shift",
     marker=(:circle,1))
hline!([ls_theo_l/ω_b_0], label="Theoretical shift", color=:red, linestyle=:dash, linewidth=1)
plot!(xlabel = "Number of oscillators in the chain", ylabel = string("Lamb shift ",L"\Delta\omega/\omega_{sys}"), legend=:bottomleft)

p2=plot(100:50:2000, abs.(Lamb_shift_l_n.-ls_theo_l)/(ls_theo_l), linewidth=.5, label="",
     marker=(:circle,1))
plot!(xlabel = "Number of oscillators in the chain", ylabel = "Relative error on Lamb shift", yaxis=:log10)

plot(p1,p2, layout=(1,2), size=(2*size_px[1],size_px[2]))
#savefig("Lamb_shift_n.pdf")

## Distribution of the onsite energies as a function of the cut-off

In [ ]:
n_huge=2000
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=90))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
ω_cut_off_list = range(20, 90, length=201)
onsite_energies_list = Vector{Vector{Float64}}(undef, length(ω_cut_off_list))
for i in ProgressBar(1:length(ω_cut_off_list)) #wrap any iterator
    ω_cut_off = ω_cut_off_list[i]
    AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=ω_cut_off))
    onsite_energies_list[i],_,_ = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
end


In [ ]:
p1=plot(ω_cut_off_list./2pi,getindex.(onsite_energies_list,1)./2pi, label="")
for i in 2:length(ω_cut_off_list)
    plot!(ω_cut_off_list./2pi,getindex.(onsite_energies_list,i)./2pi, label="")
end
plot!(xlabel = L"ω_c/2\pi"*" (GHz)", ylabel = "On-site energies "*L"\omega_i/2\pi"*" (GHz)", legend=:bottomright)
plot(p1)
#savefig("On-site_energies_VS_cut_off_exp_decay.pdf")

In [ ]:
onsite_energies_list[101]